# Montagem do dataset_final_treino_v2

**Versão:** 2  
**Criado em:** 2026-05-18  

## Objetivo
Montar o dataset de treino v2 a partir dos arquivos curated existentes, preservando integralmente o v1.
Gera duas saídas: `v2_full` (desbalanceado, documentado) e `v2_balanced` (1:1, recomendado para baseline).

## Constraints
- `dataset_final_treino_v1.csv` nunca é lido nem alterado por este notebook
- Nenhum arquivo curated antigo é sobrescrito
- Nenhum modelo é treinado aqui
- Saídas geradas em `dados/dataset_unificado/final/` com timestamp no nome

## Schema v2
Superset do v1, adiciona `origem_qualidade` (ROTULO_FORTE | ROTULO_ASSUMIDO | ROTULO_ACADEMICO).

## Fontes
| Segmento | label | label_detalhe | origem_qualidade |
|---|---|---|---|
| GFC FALSO/ENGANOSO/FORA_DE_CONTEXTO | 0 | idem | ROTULO_FORTE |
| GFC_VERDADEIRO (auditado, 23 registros) | 1 | GFC_VERDADEIRO | ROTULO_FORTE |
| RSS portais jornalísticos (titulo/titulo_resumo) | 1 | NOTICIA_REAL | ROTULO_ASSUMIDO |

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

SEED = 42
np.random.seed(SEED)

# --- Detecção robusta da raiz do projeto ---
def _find_project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / 'dados').is_dir():
        return cwd
    if (cwd.parent / 'dados').is_dir():
        return cwd.parent
    raise FileNotFoundError(
        f"Pasta 'dados' não encontrada em '{cwd}' nem em '{cwd.parent}'.\n"
        "Execute o notebook a partir da raiz do projeto ou de src/."
    )

PROJECT_ROOT = _find_project_root()

print(f"CWD         : {Path.cwd()}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

# --- Schema target ---
COLUNAS_SCHEMA = [
    'id_registro', 'texto_principal', 'label', 'label_detalhe',
    'pipeline_origem', 'portal_origem', 'origem_texto', 'origem_qualidade',
    'tamanho_chars', 'data_publicacao', 'url_origem'
]

# Aceita ROTULO_ACADEMICO mesmo sem uso nesta versão (prepara para FakeBR/FakeRecogna)
VALORES_ORIGEM_QUALIDADE = {'ROTULO_FORTE', 'ROTULO_ASSUMIDO', 'ROTULO_ACADEMICO'}

# --- Targets para v2_full (label=0 estratificado) ---
TARGET_FALSO       = 390
TARGET_ENGANOSO    = 150
TARGET_FORA_CTX    =  60
TARGET_LABEL0_FULL = TARGET_FALSO + TARGET_ENGANOSO + TARGET_FORA_CTX  # 600

# --- Cap por portal (RSS) ---
CAP_PORTAL = 120

# --- GFC_VERDADEIRO: lista de exclusão por auditoria manual ---
# Registros excluídos: [01] título explicativo, [05] evento antigo sem contexto,
# [07] comparação ambígua, [20][24] Portugal/internacional,
# [21] título de artigo (sem claim factual), [25] dependente do artigo completo.
URLS_EXCLUIR_GFC_VER = {
    "https://noticias.uol.com.br/confere/ultimas-noticias/2022/06/27/contagem-manual-de-voto-impresso-foi-discutida-e-barrada-na-camara.htm",
    "https://www1.folha.uol.com.br/poder/2023/07/tuites-em-que-flavio-dino-critica-o-sistema-eleitoral-sao-reais-mas-antigos.shtml",
    "https://noticias.uol.com.br/comprova/ultimas-noticias/2024/04/18/entenda-a-diferenca-entre-a-liberdade-de-expressao-no-brasil-e-nos-eua.htm",
    "https://observador.pt/factchecks/fact-check-subida-de-40-euros-no-salario-minimo-e-o-maior-aumento-de-sempre-como-disse-antonio-costa/",
    "https://noticias.uol.com.br/comprova/ultimas-noticias/2025/09/04/como-a-modernizacao-de-cadastros-da-reforma-tributaria-pode-reajustar-iptu.htm",
    "https://observador.pt/factchecks/fact-check-nacionalizacoes-propostas-pelo-bloco-custam-duas-bazucas-como-diz-antonio-costa/",
    "https://projetocomprova.com.br/publica%C3%A7%C3%B5es/sao-verdadeiros-exemplos-de-paises-com-protocolos-para-cloroquina-e-cannabis-em-tuite-de-deputado/",
}
# Near-duplicate de [18]: mesmo claim, forma por extenso. Remove [31].
FRAGMENTO_NEAR_DUP = "três por cento da população mundial"

print(f"  SEED = {SEED}")
print(f"  TARGET label=0 (full): {TARGET_LABEL0_FULL}")
print(f"  CAP_PORTAL: {CAP_PORTAL}")
print(f"  URLs excluídas GFC_VER: {len(URLS_EXCLUIR_GFC_VER)}")


CWD         : c:\Users\offan\Desktop\ml-checkai\src
PROJECT_ROOT: c:\Users\offan\Desktop\ml-checkai
  SEED = 42
  TARGET label=0 (full): 600
  CAP_PORTAL: 120
  URLs excluídas GFC_VER: 7


## Seção 1 — GFC Negatives (label=0)

Fonte: arquivo curated GFC mais recente (por mtime).  
Categorias incluídas: FALSO, ENGANOSO, FORA_DE_CONTEXTO.  
Amostragem estratificada por categoria com `random_state=SEED`.

In [4]:
def latest_file(glob_path: Path) -> Path:
    """Retorna o arquivo mais recente (mtime) que bate com o padrão glob de Path.

    Uso: latest_file(PROJECT_ROOT / "dados" / "subdir" / "*.csv")
    """
    files = list(glob_path.parent.glob(glob_path.name))
    if not files:
        raise FileNotFoundError(f"Nenhum arquivo encontrado: {glob_path}")
    return max(files, key=lambda p: p.stat().st_mtime)

GFC_CURATED_DIR = PROJECT_ROOT / "dados" / "pipeline_falso_google_factcheck" / "curated"
GFC_FILE = latest_file(GFC_CURATED_DIR / "*.csv")

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"GFC_CURATED_DIR: {GFC_CURATED_DIR}")
print(f"Arquivo GFC curated selecionado: {GFC_FILE.name}")

df_gfc_raw = pd.read_csv(GFC_FILE, encoding='utf-8')
print(f"Shape: {df_gfc_raw.shape}")
print("\navaliacao_categoria:")
print(df_gfc_raw['avaliacao_categoria'].value_counts().to_string())


PROJECT_ROOT  : c:\Users\offan\Desktop\ml-checkai
GFC_CURATED_DIR: c:\Users\offan\Desktop\ml-checkai\dados\pipeline_falso_google_factcheck\curated
Arquivo GFC curated selecionado: google_factcheck_curated_2026-05-18_22-40-59.csv
Shape: (3763, 13)

avaliacao_categoria:
avaliacao_categoria
FALSO               2651
ENGANOSO             936
FORA_DE_CONTEXTO     101
OUTRO                 39
VERDADEIRO            31
IMPRECISO              5


In [5]:
CATS_NEGATIVAS = ['FALSO', 'ENGANOSO', 'FORA_DE_CONTEXTO']
df_neg_pool = df_gfc_raw[df_gfc_raw['avaliacao_categoria'].isin(CATS_NEGATIVAS)].copy()
print(f"Pool label=0 disponível: {len(df_neg_pool)}")
print(df_neg_pool['avaliacao_categoria'].value_counts().to_string())

targets_neg = {
    'FALSO':           TARGET_FALSO,
    'ENGANOSO':        TARGET_ENGANOSO,
    'FORA_DE_CONTEXTO': TARGET_FORA_CTX,
}

partes_neg = []
print("\nAmostragem estratificada:")
for cat, n_target in targets_neg.items():
    subset = df_neg_pool[df_neg_pool['avaliacao_categoria'] == cat]
    n_sample = min(n_target, len(subset))
    partes_neg.append(subset.sample(n=n_sample, random_state=SEED))
    print(f"  {cat}: {n_sample} selecionados de {len(subset)} disponíveis")

df_neg = pd.concat(partes_neg, ignore_index=True)
print(f"\nTotal GFC negatives selecionados: {len(df_neg)}")

Pool label=0 disponível: 3688
avaliacao_categoria
FALSO               2651
ENGANOSO             936
FORA_DE_CONTEXTO     101

Amostragem estratificada:
  FALSO: 390 selecionados de 2651 disponíveis
  ENGANOSO: 150 selecionados de 936 disponíveis
  FORA_DE_CONTEXTO: 60 selecionados de 101 disponíveis

Total GFC negatives selecionados: 600


## Seção 2 — GFC_VERDADEIRO (label=1)

Fonte: mesmo arquivo curated GFC.  
Filtros obrigatórios:
1. Exclusão por URL (7 registros — auditoria manual: Portugal/internacional, comparação ambígua, título explicativo, claim dependente de artigo)
2. Exclusão por fragmento de texto (1 near-duplicate de [18])

Esperado: **23 registros** após filtros.

In [6]:
df_ver_pool = df_gfc_raw[df_gfc_raw['avaliacao_categoria'] == 'VERDADEIRO'].copy()
print(f"VERDADEIRO antes de filtros: {len(df_ver_pool)}")

mask_url  = ~df_ver_pool['url_origem'].isin(URLS_EXCLUIR_GFC_VER)
mask_frag = ~df_ver_pool['texto_principal'].str.contains(FRAGMENTO_NEAR_DUP, na=False, case=False)

n_excl_url  = (~mask_url).sum()
n_excl_frag = (mask_url & ~mask_frag).sum()

df_ver = df_ver_pool[mask_url & mask_frag].copy()

print(f"Excluídos por URL (auditoria): {n_excl_url}")
print(f"Excluídos por fragmento near-dup: {n_excl_frag}")
print(f"GFC_VERDADEIRO após filtros: {len(df_ver)}")

VERDADEIRO antes de filtros: 31
Excluídos por URL (auditoria): 7
Excluídos por fragmento near-dup: 1
GFC_VERDADEIRO após filtros: 23


In [7]:
assert len(df_ver) == 23, (
    f"Esperado 23 GFC_VERDADEIRO após auditoria, obtido {len(df_ver)}.\n"
    "Se o arquivo curated mudou, revisar URLS_EXCLUIR_GFC_VER e FRAGMENTO_NEAR_DUP."
)
print("OK: 23 GFC_VERDADEIRO limpos confirmados")
print("\nRegistros mantidos:")
print(df_ver[['url_origem', 'texto_principal']].to_string())

OK: 23 GFC_VERDADEIRO limpos confirmados

Registros mantidos:
                                                                                                                                                    url_origem                                                                                                                                                                                                                                                                                                                                                          texto_principal
1221                                                                                                      https://www.aosfatos.org/noticias/bolsonaro-no-flow/                                                                                                                                                                                                                                                                    

## Seção 3 — RSS Notícias Reais (label=1)

Fonte: todos os arquivos curated em `dados/pipeline_noticias_reais/curated/`.  
Filtros:
- Deduplicação por `url_origem`
- Apenas `origem_texto` ∈ {`titulo`, `titulo_resumo`} (textos curtos adequados como claim)
- Cap de `CAP_PORTAL` registros por portal (evita dominância futura de fonte única)

In [8]:
RSS_CURATED_DIR = PROJECT_ROOT / "dados" / "pipeline_noticias_reais" / "curated"
RSS_FILES = sorted(RSS_CURATED_DIR.glob("*.csv"))
assert RSS_FILES, f"Nenhum arquivo RSS curated encontrado em {RSS_CURATED_DIR}"

print(f"RSS_CURATED_DIR: {RSS_CURATED_DIR}")
print(f"Arquivos RSS curated encontrados: {len(RSS_FILES)}")
for f in RSS_FILES:
    print(f"  {f.name}")

dfs_rss = [pd.read_csv(f, encoding='utf-8') for f in RSS_FILES]
df_rss_raw = pd.concat(dfs_rss, ignore_index=True)
df_rss_dedup = df_rss_raw.drop_duplicates(subset='url_origem').copy()

print(f"\nRSS: {len(df_rss_raw)} bruto -> {len(df_rss_dedup)} após dedup por url_origem")
print("\norigem_texto:")
print(df_rss_dedup['origem_texto'].value_counts().to_string())


RSS_CURATED_DIR: c:\Users\offan\Desktop\ml-checkai\dados\pipeline_noticias_reais\curated
Arquivos RSS curated encontrados: 4
  rss_noticias_curated_2026-05-16_21-15-51.csv
  rss_noticias_curated_2026-05-17_02-46-27.csv
  rss_noticias_curated_2026-05-17_03-24-13.csv
  rss_noticias_curated_2026-05-17_03-27-03.csv

RSS: 2000 bruto -> 617 após dedup por url_origem

origem_texto:
origem_texto
titulo_resumo    219
titulo            12


In [9]:
df_rss_short = df_rss_dedup[
    df_rss_dedup['origem_texto'].isin(['titulo', 'titulo_resumo'])
].copy()
print(f"RSS textos curtos (titulo/titulo_resumo): {len(df_rss_short)}")
print("\nDistribuição por portal:")
print(df_rss_short['portal'].value_counts().to_string())

# Cap por portal
partes_rss = []
cap_aplicado = False
for portal, grupo in df_rss_short.groupby('portal'):
    n = min(CAP_PORTAL, len(grupo))
    if n < len(grupo):
        print(f"  Cap aplicado: {portal} {len(grupo)} -> {n}")
        cap_aplicado = True
    partes_rss.append(grupo.sample(n=n, random_state=SEED))

df_rss = pd.concat(partes_rss, ignore_index=True)
if not cap_aplicado:
    print(f"  Nenhum portal excedeu o cap de {CAP_PORTAL}")
print(f"\nRSS selecionados: {len(df_rss)}")

RSS textos curtos (titulo/titulo_resumo): 231

Distribuição por portal:
portal
FOLHA_PODER            100
CORREIO_BRAZILIENSE     30
VEJA_POLITICA           20
METROPOLES              20
CARTACAPITAL            20
CONGRESSO_EM_FOCO       20
PODER360                10
UOL_NOTICIAS             6
G1_POLITICA              3
BBC_BRASIL               2
  Nenhum portal excedeu o cap de 120

RSS selecionados: 231


## Seção 4 — Montagem

Mapeia colunas de origem para o schema v2, concatena os três segmentos,
valida integridade e atribui `id_registro` sequencial por prefixo.

In [10]:
def build_gfc_neg(df):
    return pd.DataFrame({
        'texto_principal': df['texto_principal'].values,
        'label':           0,
        'label_detalhe':   df['avaliacao_categoria'].values,
        'pipeline_origem': df['pipeline'].values,
        'portal_origem':   df['fonte'].values,
        'origem_texto':    df['tipo_conteudo'].values,
        'origem_qualidade':'ROTULO_FORTE',
        'tamanho_chars':   df['texto_principal'].str.len().values,
        'data_publicacao': df['data_publicacao'].values,
        'url_origem':      df['url_origem'].values,
    })

def build_gfc_ver(df):
    return pd.DataFrame({
        'texto_principal': df['texto_principal'].values,
        'label':           1,
        'label_detalhe':   'GFC_VERDADEIRO',
        'pipeline_origem': df['pipeline'].values,
        'portal_origem':   df['fonte'].values,
        'origem_texto':    df['tipo_conteudo'].values,
        'origem_qualidade':'ROTULO_FORTE',
        'tamanho_chars':   df['texto_principal'].str.len().values,
        'data_publicacao': df['data_publicacao'].values,
        'url_origem':      df['url_origem'].values,
    })

def build_rss(df):
    return pd.DataFrame({
        'texto_principal': df['texto_principal'].values,
        'label':           1,
        'label_detalhe':   'NOTICIA_REAL',
        'pipeline_origem': df['pipeline'].values,
        'portal_origem':   df['portal'].values,
        'origem_texto':    df['origem_texto'].values,
        'origem_qualidade':'ROTULO_ASSUMIDO',
        'tamanho_chars':   df['texto_principal'].str.len().values,
        'data_publicacao': df['data_publicacao'].values,
        'url_origem':      df['url_origem'].values,
    })

part_neg = build_gfc_neg(df_neg)
part_ver = build_gfc_ver(df_ver)
part_rss = build_rss(df_rss)

print("Partes montadas:")
print(f"  GFC negatives (label=0): {len(part_neg)}")
print(f"  GFC_VERDADEIRO (label=1): {len(part_ver)}")
print(f"  RSS (label=1):  {len(part_rss)}")
print(f"  Total label=1:  {len(part_ver) + len(part_rss)}")

Partes montadas:
  GFC negatives (label=0): 600
  GFC_VERDADEIRO (label=1): 23
  RSS (label=1):  231
  Total label=1:  254


In [11]:
def gerar_ids(df):
    """Atribui id_registro sequencial por prefixo baseado em label_detalhe."""
    prefixo_map = {
        'FALSO':            'GFC_NEG',
        'ENGANOSO':         'GFC_NEG',
        'FORA_DE_CONTEXTO': 'GFC_NEG',
        'GFC_VERDADEIRO':   'GFC_VER',
        'NOTICIA_REAL':     'RSS',
    }
    contadores = {}
    ids = []
    for det in df['label_detalhe']:
        p = prefixo_map.get(det, 'UNK')
        contadores[p] = contadores.get(p, 0) + 1
        ids.append(f"{p}_{contadores[p]:05d}")
    df = df.copy()
    df['id_registro'] = ids
    return df

# Concat full antes das validações
df_full_raw = pd.concat([part_neg, part_ver, part_rss], ignore_index=True)

# Validações de integridade
assert df_full_raw['texto_principal'].notna().all(), "texto_principal com nulos"
assert df_full_raw['label'].isin([0, 1]).all(), "label fora de {0, 1}"
assert df_full_raw['origem_qualidade'].isin(VALORES_ORIGEM_QUALIDADE).all(), "origem_qualidade inválido"
assert len(df_full_raw[df_full_raw['label'] == 0]) > 0, "Sem registros label=0"
assert len(df_full_raw[df_full_raw['label'] == 1]) > 0, "Sem registros label=1"
assert df_full_raw['url_origem'].notna().all(), "url_origem com nulos"

df_full_raw = gerar_ids(df_full_raw)
assert df_full_raw['id_registro'].is_unique, "id_registro com duplicatas"

df_full_raw = df_full_raw[COLUNAS_SCHEMA]

print("Validações: OK")
print(f"Dataset base: {df_full_raw.shape}")
print("\nDistribuição:")
print(df_full_raw['label'].value_counts().to_string())
print("\nlabel_detalhe:")
print(df_full_raw['label_detalhe'].value_counts().to_string())

Validações: OK
Dataset base: (854, 11)

Distribuição:
label
0    600
1    254

label_detalhe:
label_detalhe
FALSO               390
NOTICIA_REAL        231
ENGANOSO            150
FORA_DE_CONTEXTO     60
GFC_VERDADEIRO       23


## Seção 5 — Versões de saída

### v2_full
Preserva toda a seleção (600 label=0, ~254 label=1). Desbalanceado.  
**Não recomendado para treino direto** sem `class_weight='balanced'` ou reamostragem.

### v2_balanced
Corte de label=0 para igualar label=1 (sem oversampling, sem SMOTE).  
Amostragem estratificada por `label_detalhe` com `random_state=SEED`.  
**Recomendado para baseline.**

In [12]:
df_v2_full = df_full_raw.copy()

n0_full = (df_v2_full['label'] == 0).sum()
n1_full = (df_v2_full['label'] == 1).sum()
ratio   = n0_full / n1_full if n1_full else float('inf')

print("=== v2_FULL ===")
print(f"Total: {len(df_v2_full)}")
print(f"label=0: {n0_full} | label=1: {n1_full} | ratio: {ratio:.2f}")
print("\nAVISO: v2_full está desbalanceado.")
print("       Não recomendado para treino direto sem class_weight ou reamostragem.")
print("       Use v2_balanced para baseline.")

=== v2_FULL ===
Total: 854
label=0: 600 | label=1: 254 | ratio: 2.36

AVISO: v2_full está desbalanceado.
       Não recomendado para treino direto sem class_weight ou reamostragem.
       Use v2_balanced para baseline.


In [13]:
label1_df = df_full_raw[df_full_raw['label'] == 1].copy()
label0_df = df_full_raw[df_full_raw['label'] == 0].copy()
n_bal     = len(label1_df)

# Proporções de label_detalhe no pool label=0 (para manter distribuição)
props = label0_df['label_detalhe'].value_counts(normalize=True)
cats  = list(props.index)

partes_bal = []
alocado    = 0
print("Amostragem estratificada label=0 para balanced:")
for i, cat in enumerate(cats):
    subset = label0_df[label0_df['label_detalhe'] == cat]
    if i == len(cats) - 1:
        # Última categoria absorve resto para garantir total exato
        n_cat = n_bal - alocado
    else:
        n_cat = round(n_bal * props[cat])
    n_cat = min(n_cat, len(subset))
    n_cat = max(n_cat, 0)
    partes_bal.append(subset.sample(n=n_cat, random_state=SEED))
    alocado += n_cat
    print(f"  {cat}: {n_cat}")

label0_bal = pd.concat(partes_bal, ignore_index=True)

df_v2_balanced = (
    pd.concat([label0_bal, label1_df], ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

# Reatribui IDs (ordem embaralhada)
df_v2_balanced = gerar_ids(df_v2_balanced)
assert df_v2_balanced['id_registro'].is_unique
df_v2_balanced = df_v2_balanced[COLUNAS_SCHEMA]

n0_bal = (df_v2_balanced['label'] == 0).sum()
n1_bal = (df_v2_balanced['label'] == 1).sum()
print(f"\n=== v2_BALANCED ===")
print(f"Total: {len(df_v2_balanced)}")
print(f"label=0: {n0_bal} | label=1: {n1_bal}")
print(df_v2_balanced['label_detalhe'].value_counts().to_string())

Amostragem estratificada label=0 para balanced:
  FALSO: 165
  ENGANOSO: 64
  FORA_DE_CONTEXTO: 25

=== v2_BALANCED ===
Total: 508
label=0: 254 | label=1: 254
label_detalhe
NOTICIA_REAL        231
FALSO               165
ENGANOSO             64
FORA_DE_CONTEXTO     25
GFC_VERDADEIRO       23


In [14]:
TS      = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
OUT_DIR = PROJECT_ROOT / "dados" / "dataset_unificado" / "final"
OUT_DIR.mkdir(parents=True, exist_ok=True)

file_full = OUT_DIR / f"dataset_final_treino_v2_full_{TS}.csv"
file_bal  = OUT_DIR / f"dataset_final_treino_v2_balanced_{TS}.csv"

df_v2_full.to_csv(file_full, index=False, encoding='utf-8')
df_v2_balanced.to_csv(file_bal,  index=False, encoding='utf-8')

print(f"Salvo: {file_full}")
print(f"Salvo: {file_bal}")


Salvo: c:\Users\offan\Desktop\ml-checkai\dados\dataset_unificado\final\dataset_final_treino_v2_full_2026-05-19_00-48-15.csv
Salvo: c:\Users\offan\Desktop\ml-checkai\dados\dataset_unificado\final\dataset_final_treino_v2_balanced_2026-05-19_00-48-15.csv


In [15]:
SEP = "=" * 65

print(SEP)
print("RELATORIO FINAL -- dataset_final_treino_v2")
print(SEP)

for nome, df in [("v2_FULL", df_v2_full), ("v2_BALANCED", df_v2_balanced)]:
    n0 = (df['label'] == 0).sum()
    n1 = (df['label'] == 1).sum()
    print(f"\n{'--- ' + nome + ' ---':^65}")
    print(f"Total registros: {len(df)}")
    print(f"Ratio label=0/label=1: {n0/n1:.2f}" if n1 else "")

    print("\n[Por label]")
    print(df['label'].value_counts().rename({0: 'label=0 (fake/misleading)',
                                              1: 'label=1 (real/verdadeiro)'}).to_string())

    print("\n[Por label_detalhe]")
    print(df['label_detalhe'].value_counts().to_string())

    print("\n[Por origem_qualidade]")
    print(df['origem_qualidade'].value_counts().to_string())

    print("\n[Por pipeline_origem]")
    print(df['pipeline_origem'].value_counts().to_string())

    print("\n[Por portal_origem (top 20)]")
    print(df['portal_origem'].value_counts().head(20).to_string())

    print("\n[Tamanho texto por label]")
    for lbl in [0, 1]:
        s = df[df['label'] == lbl]['tamanho_chars']
        print(f"  label={lbl}: media={s.mean():.0f}  mediana={s.median():.0f}  "
              f"min={s.min()}  max={s.max()}")

    if nome == "v2_FULL":
        print("\n[!] AVISO: v2_FULL NAO recomendado para treino direto.")
        print("    Use class_weight='balanced' no modelo ou prefira v2_BALANCED.")

print(f"\n{SEP}")
print("Arquivos gerados:")
print(f"  {file_full.name}")
print(f"  {file_bal.name}")
print(SEP)

RELATORIO FINAL -- dataset_final_treino_v2

                         --- v2_FULL ---                         
Total registros: 854
Ratio label=0/label=1: 2.36

[Por label]
label
label=0 (fake/misleading)    600
label=1 (real/verdadeiro)    254

[Por label_detalhe]
label_detalhe
FALSO               390
NOTICIA_REAL        231
ENGANOSO            150
FORA_DE_CONTEXTO     60
GFC_VERDADEIRO       23

[Por origem_qualidade]
origem_qualidade
ROTULO_FORTE       623
ROTULO_ASSUMIDO    231

[Por pipeline_origem]
pipeline_origem
google_factcheck    623
noticias_reais      231

[Por portal_origem (top 20)]
portal_origem
AOS_FATOS              164
ESTADÃO                140
FOLHA_PODER            100
UOL_NOTÍCIAS            85
AFP_CHECAMOS            81
BOATOS.ORG              44
PROJETO_COMPROVA        31
CORREIO_BRAZILIENSE     30
OBSERVADOR              25
METROPOLES              20
CONGRESSO_EM_FOCO       20
CARTACAPITAL            20
VEJA_POLITICA           20
DESCONHECIDA            20
BOL_-